# Chapter 6 &mdash; The Language Stethoscope: Indistinguishability

**Concept 6 of the Chapter 6 decomposition:** *The Language Stethoscope: Indistinguishability and Minimality*

The language of a <i>state</i> is what it accepts as a start state; states with the same one can merge.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Stethoscope/Concept-Language-Stethoscope.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Put a stethoscope on a single state $q$: **the language of $q$** is what the machine
would accept if $q$ were the start state.

Two states are **indistinguishable** if their languages are equal &mdash; no string can
tell them apart from where they stand. Such states can be **merged** without changing
the machine's language, and a DFA is **minimal** exactly when no two states are
indistinguishable.

The finite test: instead of comparing infinite languages, compare behaviour on all
strings up to length $|Q|-1$, which turns out to be enough (Concept 8).

## 2. Definitions

### A machine with redundancy built in

In [ ]:
D = md2mc('''DFA
IF : 0 -> A
IF : 1 -> B
A  : 0 -> IF
A  : 1 -> C
B  : 0 -> C
B  : 1 -> IF
C  : 0 -> B
C  : 1 -> A
''')

### The language of a state, up to a length bound

In [ ]:
from itertools import product
def lang_of_state(D, q, n):
    out = set()
    for k in range(n+1):
        for p in product(sorted(D["Sigma"]), repeat=k):
            s = ''.join(p)
            if run_dfa_h(D, s, q) in D["F"]:
                out.add(s)
    return out

### Indistinguishability, decided by comparing those languages

In [ ]:
def indist_pairs(D, n=None):
    n = n if n is not None else len(D["Q"])
    L = {q: lang_of_state(D, q, n) for q in D["Q"]}
    qs = sorted(D["Q"])
    return [(a, b) for i, a in enumerate(qs) for b in qs[i+1:] if L[a] == L[b]]

## 3. Tests

Each state's language, listed short.

In [ ]:
for q in sorted(D["Q"]):
    L = sorted(lang_of_state(D, q, 3), key=lambda s: (len(s), s))
    print("%-4s accepts (len<=3): %s" % (q, L[:8]))

Indistinguishable pairs, found by comparing those languages.

In [ ]:
pairs = indist_pairs(D)
print("indistinguishable pairs :", pairs)
print("so the minimal machine should have %d states; min_dfa says %d"
      % (len(D["Q"]) - len(pairs), len(min_dfa(D)["Q"])))

Merging an indistinguishable pair preserves the language &mdash; `min_dfa` does exactly that.

In [ ]:
m = min_dfa(D)
print("original %d states -> minimal %d states" % (len(D["Q"]), len(m["Q"])))
assert langeq_dfa(D, m)
print("same language after merging? ", langeq_dfa(D, m))

A **minimal** machine has no indistinguishable pairs &mdash; that is the definition.

In [ ]:
print("indistinguishable pairs in the minimal machine :", indist_pairs(m))
assert indist_pairs(m) == []

Distinguishing a pair means exhibiting a string; here is one.

In [ ]:
qs = sorted(m["Q"])
if len(qs) >= 2:
    a, b = qs[0], qs[1]
    La, Lb = lang_of_state(m, a, len(m["Q"])), lang_of_state(m, b, len(m["Q"]))
    w = sorted(La ^ Lb, key=lambda s: (len(s), s))[0]
    print("%s and %s are separated by %r : %s vs %s"
          % (a, b, w, run_dfa_h(m, w, a) in m["F"], run_dfa_h(m, w, b) in m["F"]))

## 4. Exercises


1. What is the language of a black-hole state? Of a state all of whose successors are final?
2. Show that indistinguishability is an equivalence relation.
3. Why is length $|Q|-1$ enough? (Concept 8 answers this.)

In [ ]:
# Your work for the exercises above.